In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import DATA_DIR, DATASETS, split_data, encode_and_scale, save_splits

sns.set_theme(style="whitegrid", context="paper")
FIG_DIR = ROOT / "report" / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
data = {name: load() for name, load in DATASETS.items()}

summary = pd.DataFrame(
    {
        "rows": [len(pd.read_csv(DATA_DIR / f"{name}.csv.gz", usecols=[0])) for name in data],
        "rows (clean)": [len(y) for _, y in data.values()],
    },
    index=list(data),
).round(2)
summary

,rows,rows (clean)
higgs,98050,98049
covertype,581012,581012
adult,48842,45222


In [5]:
rows = []
for name, (X, y) in data.items():
    X_train, X_cal, X_test, y_train, y_cal, y_test = split_data(X, y)
    X_train, X_cal, X_test = encode_and_scale(X_train, X_cal, X_test)
    save_splits(name, X_train, X_cal, X_test, y_train, y_cal, y_test)
    k = X.select_dtypes("number").shape[1]
    for part, Xp, yp in [("train", X_train, y_train), ("cal", X_cal, y_cal), ("test", X_test, y_test)]:
        rows.append({
            "dataset": name, "split": part, "rows": Xp.shape[0], "features": Xp.shape[1],
            "positives (%)": 100 * yp.mean(),
        })
pd.DataFrame(rows).set_index(["dataset", "split"]).round(3)

rows  features  positives (%)
dataset   split                                
higgs     train  24000        28         52.858
          cal     8000        28         52.863
          test    8000        28         52.850
covertype train  24000        53         48.758
          cal     8000        53         48.762
          test    8000        53         48.762
adult     train  24000       104         24.783
          cal     8000       104         24.788
          test    8000       104         24.788